In [15]:

# Step 1: Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import xgboost as xgb

In [16]:
# Step 2: Load Dataset
df = pd.read_csv("fertilizer_recommendation_dataset.csv")

print("First 5 rows of dataset:")
print(df.head())

First 5 rows of dataset:
   N (mg/kg)  P (mg/kg)  K (mg/kg)    pH  Moisture (%)  EC (dS/m)  \
0         43         32         22  5.38            13       1.06   
1         56         29         69  6.53            33       1.87   
2         33         43         16  7.53            34       0.71   
3         19         37         66  6.47            12       1.23   
4         47          5         45  5.34            12       0.80   

   Temperature (°C) Fertilizer_Recommendation  
0                21                       MOP  
1                26                       MOP  
2                29                       MOP  
3                26                      Urea  
4                20                       DAP  


In [17]:
# Step 3: Split features & labels
X = df.drop("Fertilizer_Recommendation", axis=1)
y = df["Fertilizer_Recommendation"]

# Encode target labels
le = LabelEncoder()
y = le.fit_transform(y)

# Optional: Scale features (good for ML models)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [18]:

# Step 4: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

In [19]:
# Step 5: Handle Class Imbalance with SMOTE (safe version)
from imblearn.over_sampling import SMOTE

try:
    smote = SMOTE(random_state=42, k_neighbors=1)  # set k_neighbors=1 to handle small classes
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    print("SMOTE applied successfully")
    print("Before SMOTE:", np.bincount(y_train))
    print("After SMOTE:", np.bincount(y_train_res))

except ValueError as e:
    print("SMOTE failed due to too few samples per class:", e)
    print("Proceeding without SMOTE...")

    # fallback: use original training data
    X_train_res, y_train_res = X_train, y_train

SMOTE applied successfully
Before SMOTE: [ 13 224   5  18 238 302]
After SMOTE: [302 302 302 302 302 302]


In [21]:

# Step 6: Define XGBoost Classifier
xgb_clf = xgb.XGBClassifier(eval_metric="mlogloss", use_label_encoder=False, random_state=42)

# Hyperparameter Grid for Tuning
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
    "subsample": [0.8, 1.0]
}

In [24]:

# Step 7: Grid Search for Best Parameters
grid_search = GridSearchCV(
    estimator=xgb_clf,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    verbose=2,
    scoring="accuracy"
)

grid_search.fit(X_train_res, y_train_res)

print("Best Parameters:", grid_search.best_params_)


Fitting 5 folds for each of 54 candidates, totalling 270 fits


c:\Users\safid\anaconda3\envs\fert\lib\site-packages\xgboost\training.py:183: UserWarning: [02:58:17] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best Parameters: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200, 'subsample': 0.8}


In [23]:
# Step 8: Evaluate on Test Set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Classification Report:
                 precision    recall  f1-score   support

Compost/Organic       0.00      0.00      0.00         3
            DAP       0.75      0.59      0.66        56
         Gypsum       0.00      0.00      0.00         1
           Lime       0.57      1.00      0.73         4
            MOP       0.62      0.67      0.64        60
           Urea       0.63      0.66      0.65        76

       accuracy                           0.64       200
      macro avg       0.43      0.49      0.45       200
   weighted avg       0.65      0.64      0.64       200


Confusion Matrix:
[[ 0  0  0  0  1  2]
 [ 1 33  0  1 10 11]
 [ 0  0  0  0  0  1]
 [ 0  0  0  4  0  0]
 [ 2  3  0  0 40 15]
 [ 2  8  0  2 14 50]]


c:\Users\safid\anaconda3\envs\fert\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\safid\anaconda3\envs\fert\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\safid\anaconda3\envs\fert\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [10]:
# Step 9: Cross-Validation Accuracy
cv_scores = cross_val_score(best_model, X_scaled, y, cv=5)
print("\nCross-Validation Accuracy:", cv_scores.mean())

c:\Users\safid\anaconda3\envs\fert\lib\site-packages\xgboost\training.py:183: UserWarning: [02:51:36] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\safid\anaconda3\envs\fert\lib\site-packages\xgboost\training.py:183: UserWarning: [02:51:36] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\safid\anaconda3\envs\fert\lib\site-packages\xgboost\training.py:183: UserWarning: [02:51:36] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\safid\anaconda3\envs\fert\lib\site-packages\xgboost\training.py:183: UserWarning: [02:51:36] WARNING: C:\b\abs_d97hy_84m6\croot\xgboo


Cross-Validation Accuracy: 0.7550000000000001


c:\Users\safid\anaconda3\envs\fert\lib\site-packages\xgboost\training.py:183: UserWarning: [02:51:36] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
